# NB_patient_360

Builds `adb_qnxthealth.gold.patient_360` — one row per patient with demographics, hospital/insurance lookups, and aggregated lab-test statistics. Full rebuild every run.

In [0]:
%sql
CREATE OR REPLACE TABLE adb_qnxthealth.gold.patient_360 AS

WITH lab_agg AS (
  SELECT
    patient_id,
    COUNT(*)                                                         AS total_tests,
    COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)          AS abnormal_test_count,
    COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)          AS critical_test_count,
    MAX(test_date)                                                   AS last_test_date
  FROM adb_qnxthealth.silver.lab_results
  GROUP BY patient_id
)

SELECT
  -- Patient demographics
  p.patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                      AS date_of_birth,
  FLOOR(DATEDIFF(CURRENT_DATE(), CAST(p.date_of_birth AS DATE)) / 365.25) AS age,
  p.gender,
  p.city,
  p.state,

  -- Hospital lookup
  p.registered_hospital_id                                           AS hospital_id,
  h.hospital_name,

  -- Insurance lookup
  p.insurance_provider_id,
  ip.provider_name                                                   AS insurance_provider_name,

  -- Lab-test aggregates (0 when no tests exist)
  COALESCE(la.total_tests, 0)                                        AS total_tests,
  COALESCE(la.abnormal_test_count, 0)                                AS abnormal_test_count,
  COALESCE(la.critical_test_count, 0)                                AS critical_test_count,
  la.last_test_date

FROM      adb_qnxthealth.silver.patients            p
LEFT JOIN adb_qnxthealth.silver.hospitals            h   ON p.registered_hospital_id = h.hospital_id
LEFT JOIN adb_qnxthealth.silver.insurance_providers  ip  ON p.insurance_provider_id   = ip.insurance_provider_id
LEFT JOIN lab_agg                                    la  ON p.patient_id              = la.patient_id;

In [0]:
%skip
%sql
SELECT count(*) AS total_rows,
       count(DISTINCT patient_id) AS unique_patients,
       avg(age) AS avg_age,
       sum(total_tests) AS total_lab_tests,
       count_if(total_tests = 0) AS patients_with_no_tests
FROM   adb_qnxthealth.gold.patient_360;

In [0]:
record_count = spark.table("adb_qnxthealth.gold.patient_360").count()
dbutils.notebook.exit(str(record_count))